# DRAGNET - deployment-scale probe at k = 50 (vLLM, free Kaggle T4)

k = 30+ OOMs a 16 GB card under transformers (a 50-passage prefill needs too much activation). vLLM's
**paged KV cache + chunked prefill** bound that memory, and **AWQ 4-bit** weights (~4.5 GB) leave the
rest for the cache, so k = 50 fits a single T4 - and runs fast (optimised prefill). Precision is not a
confound: the paper's own 4-bit-vs-fp16 check is 32 vs 31% no-culprit.

**This is a stretch goal.** W1 is already closed at k = 20; k = 50 only strengthens the number. The
one real risk is the vLLM install fighting Kaggle's torch/CUDA - see the note in cell 1 if it fails.

**Before running:** Settings -> Accelerator **GPU T4** (one card) -> Internet **on**. Budget: build a
few min, probe ~30-60 min at 50 cases.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
os.chdir(WORK)

# vLLM is installed FIRST so it pins its own torch/transformers; lineup+dragnet then go in with
# --no-deps so they cannot downgrade that stack, and only their few runtime extras are added.
# If this cell fails on a torch/CUDA mismatch, pin a known-good build, e.g.
#   pip install vllm==0.6.3.post1
# or check the vLLM release notes for the version matching Kaggle's current CUDA.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)

def fetch(name, url):
    if (WORK / name).exists():
        subprocess.run(['git', '-C', name, 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, name], check=True)

fetch('lineup', 'https://github.com/santoshcheethiralame-dot/LINEUP')
dragnet_src = next((p.parent for p in Path('/kaggle/input').glob('*/**/src/dragnet/__init__.py')), None)
if dragnet_src is not None:
    shutil.rmtree(WORK / 'dragnet', ignore_errors=True)
    shutil.copytree(dragnet_src.parent.parent, WORK / 'dragnet')
else:
    fetch('dragnet', 'https://github.com/santoshcheethiralame-dot/DRAGNET')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', './lineup', '-e', './dragnet'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rank-bm25', 'datasets'], check=True)

for pkg in ('lineup', 'dragnet'):
    src = str(WORK / pkg / 'src')
    if src not in sys.path:
        sys.path.insert(0, src)

import vllm
print('[ok] vllm', vllm.__version__, '+ lineup + dragnet ready')

In [ ]:
MODEL = 'Qwen/Qwen2.5-7B-Instruct-AWQ'   # AWQ 4-bit checkpoint; vLLM loads it directly
TAG = 'qwen'
DATASET = 'hotpotqa'          # hotpotqa | 2wiki | musique
K = 50                        # deployment depth (8x the grid's k=6). vLLM makes this fit a T4
LIMIT = 500                   # source questions; build stops once it has enough wrong cases
LIMIT_CASES = 50              # wrong cases to probe
NONMONO_SAMPLES = 4           # sampled supersets per case
MAX_MODEL_LEN = 16384         # vLLM context length; a 50-passage prompt is ~8k tokens, so 8192 is too
                              # small. 16384 covers k=50 with margin; use 32768 for k=100
SEED = 0

cell = f'runs/{DATASET}/largek{K}/{TAG}'
print(cell)

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, 'dragnet/scripts/run_largek_probe.py', '--cell', cell,
     '--dataset', DATASET, '--k', str(K), '--limit', str(LIMIT), '--seed', str(SEED),
     '--model', MODEL, '--backend', 'vllm', '--max-model-len', str(MAX_MODEL_LEN),
     '--limit-cases', str(LIMIT_CASES), '--nonmono-samples', str(NONMONO_SAMPLES)],
    check=True,
)

In [ ]:
try:
    import json
    from collections import Counter
    from math import sqrt

    rows = [json.loads(l) for l in open(f'{cell}/largek_probe.jsonl', encoding='utf-8') if l.strip()]
    ok = [r for r in rows if r['status'] == 'ok']

    def wilson(hits, n, z=1.96):
        if not n: return (0.0, 0.0)
        p, d = hits / n, 1 + z * z / n
        centre, margin = p + z * z / (2 * n), z * sqrt(p * (1 - p) / n + z * z / (4 * n * n))
        return (centre - margin) / d, (centre + margin) / d

    n = len(ok)
    print(f"k={K}  wrong probed {len(rows)}  evaluable {n}  "
          f"parametric {sum(r['status']=='parametric' for r in rows)}  "
          f"budget {sum(r['status']=='budget' for r in rows)}  "
          f"oom {sum(r['status']=='oom' for r in rows)}")
    for key, ref in (('plural', '0.86 @k20 / 0.45-0.63 @k6'),
                     ('disjoint', '0.25 @k20 / 0.28 @k6'),
                     ('nonmono', '0.63 @k20 / 0.67-0.87 @k6')):
        hits = sum(r[key] for r in ok); lo, hi = wilson(hits, n)
        print(f"  {key:8s} >= {hits/n:.2f}  [{lo:.2f}, {hi:.2f}]   (ref: {ref})")
except Exception as exc:
    print('readback skipped:', repr(exc))

In [ ]:
import shutil
stem = f'dragnet_{DATASET}_{TAG}_largek{K}_vllm_seed{SEED}'
shutil.make_archive(stem, 'zip', 'runs')
print('download:', f'/kaggle/working/{stem}.zip')